In [2]:
import numpy as np
from tenpy.models.model import CouplingMPOModel, NearestNeighborModel
from tenpy.models.lattice import Chain
from tenpy.networks.site import SpinHalfSite, SpinHalfFermionSite, set_common_charges, GroupedSite
from tenpy.networks.mps import MPS
from tenpy.algorithms import dmrg, tebd
from tenpy.networks.mpo import MPO


In [32]:
class AndersonImpurityModel(CouplingMPOModel):

    def init_sites(self, model_params):
        conserve = model_params.get('conserve', 'N')
        return SpinHalfFermionSite(cons_N=conserve)

    def init_lattice(self, model_params):
        L = model_params['L']
        site = self.init_sites(model_params)
        return Chain(L, site, bc='open')

    def init_terms(self, model_params):

        t = model_params['t']
        U = model_params['U']
        eps_d = model_params['eps_d']

        L = self.lat.N_sites
        imp = L // 2
        print(self.lat.unit_cell[0].opnames)
        for u1, u2, dx in self.lat.pairs['nearest_neighbors']:
            self.add_coupling(-1.0 * t, u1, 'Cdu', u2, 'Cu', dx, plus_hc=True)
            self.add_coupling(-1.0 * t, u1, 'Cdd', u2, 'Cd', dx, plus_hc=True)

        # impurity onsite energy
        self.add_onsite_term(eps_d, imp, 'Ntot')

        # impurity Hubbard interaction
        self.add_onsite_term(U, imp, 'NuNd')

In [33]:
model_params = {
    'L': 40,
    't': 1.0,
    'U': 8.0,
    'eps_d': -4.0,      # particle-hole symmetric point
    'conserve': 'N'
}

model = AndersonImpurityModel(model_params)

{'Nu', 'NuNd', 'Sz', 'Sm', 'dN', 'Ntot', 'Cdu', 'Sp', 'Nd', 'Cd', 'Cdd', 'JWu', 'Cu', 'Id', 'JW', 'JWd'}
